In [ ]:
import os
import sys
import random

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from src.data_utils import get_mnist_dataloaders, get_class_names
from src.models import SimpleCNN
from src.evaluate import (
    collect_predictions_with_probs,
    extract_feature_vectors,
    find_most_confused_pairs,
    compute_classification_metrics,
)
from src.visualize import (
    plot_confusion_matrix,
    visualize_conv1_kernels,
    visualize_feature_maps,
    show_misclassified_examples,
    visualize_class_prototypes,
    plot_feature_embedding_2d,
    plot_class_centroid_distance_heatmap,
    show_confused_pair_examples,
)

In [ ]:
SEED = 42
BATCH_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RESULT_FIG_DIR = "../results/figures/g3_feature_visualization"
RESULT_TABLE_DIR = "../results/tables"
os.makedirs(RESULT_FIG_DIR, exist_ok=True)
os.makedirs(RESULT_TABLE_DIR, exist_ok=True)

print("DEVICE:", DEVICE)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
class_names = get_class_names()

In [ ]:
train_loader, val_loader, test_loader = get_mnist_dataloaders(
    data_dir="../data",
    batch_size=BATCH_SIZE,
    val_size=5000,
    num_workers=0,
    augment=False,
    normalize=False,
    seed=SEED,
)

In [ ]:
model = SimpleCNN().to(DEVICE)

ckpt_path = "../results/logs/g1_best_cnn.pt"
state_dict = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

print("Loaded checkpoint from:", ckpt_path)

In [ ]:
pred_result = collect_predictions_with_probs(model, test_loader, DEVICE)

images = pred_result["images"]
labels = pred_result["labels"]
preds = pred_result["preds"]
probs = pred_result["probs"]
logits = pred_result["logits"]

metrics = compute_classification_metrics(labels, preds, class_names=class_names)

print("Accuracy        :", f"{metrics['accuracy']:.4f}")
print("Macro Precision :", f"{metrics['macro_precision']:.4f}")
print("Macro Recall    :", f"{metrics['macro_recall']:.4f}")
print("Macro F1        :", f"{metrics['macro_f1']:.4f}")
print()
print(metrics["classification_report"])

In [ ]:
plot_confusion_matrix(
    metrics["confusion_matrix"],
    class_names=class_names,
    save_path=os.path.join(RESULT_FIG_DIR, "g3_confusion_matrix.png"),
)

In [ ]:
visualize_conv1_kernels(
    model,
    save_path=os.path.join(RESULT_FIG_DIR, "g3_conv1_kernels.png"),
)

In [ ]:
chosen_indices = [0, 1, 2, 3]

for idx in chosen_indices:
    visualize_feature_maps(
        model,
        torch.tensor(images[idx]),
        device=DEVICE,
        layer_name="conv1",
        max_channels=8,
        save_path=os.path.join(RESULT_FIG_DIR, f"sample_{idx}_conv1_maps.png"),
    )

    visualize_feature_maps(
        model,
        torch.tensor(images[idx]),
        device=DEVICE,
        layer_name="conv2",
        max_channels=8,
        save_path=os.path.join(RESULT_FIG_DIR, f"sample_{idx}_conv2_maps.png"),
    )

In [ ]:
visualize_class_prototypes(
    images=images,
    labels=labels,
    probs=probs,
    n_classes=10,
    top_k=8,
    save_dir=os.path.join(RESULT_FIG_DIR, "class_prototypes"),
)

In [ ]:
show_misclassified_examples(
    model,
    test_loader,
    device=DEVICE,
    n=16,
    save_path=os.path.join(RESULT_FIG_DIR, "g3_misclassified_examples.png"),
)

In [ ]:
pairs = find_most_confused_pairs(metrics["confusion_matrix"], top_k=5)
pairs_df = pd.DataFrame(pairs, columns=["true_class", "pred_class", "count"])
pairs_df

In [ ]:
for true_c, pred_c, count in pairs:
    print(f"true={true_c}, pred={pred_c}, count={count}")
    show_confused_pair_examples(
        images=images,
        labels=labels,
        preds=preds,
        probs=probs,
        true_class=true_c,
        pred_class=pred_c,
        top_k=8,
        save_path=os.path.join(
            RESULT_FIG_DIR,
            f"confused_true_{true_c}_pred_{pred_c}.png"
        ),
    )

In [ ]:
feature_result = extract_feature_vectors(
    model,
    test_loader,
    DEVICE,
    feature_key="pool2",
)

features = feature_result["features"]
feature_labels = feature_result["labels"]

print("features shape:", features.shape)

In [ ]:
pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(features)

plot_feature_embedding_2d(
    pca_2d,
    feature_labels,
    title="PCA of CNN Learned Features",
    save_path=os.path.join(RESULT_FIG_DIR, "g3_feature_pca.png"),
)

In [ ]:
subset_n = 3000
subset_idx = np.random.choice(len(features), size=subset_n, replace=False)

tsne_2d = TSNE(
    n_components=2,
    random_state=SEED,
    init="pca",
    learning_rate="auto",
    perplexity=30,
).fit_transform(features[subset_idx])

plot_feature_embedding_2d(
    tsne_2d,
    feature_labels[subset_idx],
    title="t-SNE of CNN Learned Features",
    save_path=os.path.join(RESULT_FIG_DIR, "g3_feature_tsne.png"),
)

In [ ]:
plot_class_centroid_distance_heatmap(
    features,
    feature_labels,
    n_classes=10,
    save_path=os.path.join(RESULT_FIG_DIR, "g3_centroid_distance_heatmap.png"),
)

In [ ]:
summary = {
    "accuracy": metrics["accuracy"],
    "macro_precision": metrics["macro_precision"],
    "macro_recall": metrics["macro_recall"],
    "macro_f1": metrics["macro_f1"],
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(
    os.path.join(RESULT_TABLE_DIR, "g3_visualization_summary.csv"),
    index=False,
)

pairs_df.to_csv(
    os.path.join(RESULT_TABLE_DIR, "g3_top_confused_pairs.csv"),
    index=False,
)

print("Saved summary tables.")
summary_df

In [ ]:
top_pair = pairs[0] if len(pairs) > 0 else None

text = []
text.append("G3 特征可视化与误差分析结论草稿：")
text.append(f"1. 模型测试准确率为 {metrics['accuracy']:.4f}，Macro-F1 为 {metrics['macro_f1']:.4f}。")
text.append("2. 第一层卷积核可视化表明，网络低层主要学习局部边缘、方向变化与笔画纹理等基础模式。")
text.append("3. 特征图可视化显示，随着层数加深，网络响应由局部边缘逐渐过渡到更抽象的数字结构。")
text.append("4. 各类别高置信样本可视为模型学习到的“类别原型”，说明模型对典型书写形态具有较强稳定识别能力。")
if top_pair is not None:
    text.append(f"5. 最显著混淆类别之一为真类 {top_pair[0]} 被预测为 {top_pair[1]}，反映出相近数字在局部结构上的相似性。")
text.append("6. PCA/t-SNE 嵌入显示不同数字类别在高层特征空间中总体可分，但邻近类别之间仍存在局部重叠。")
text.append("7. 类中心距离热图进一步说明，易混淆类别在特征空间中的几何距离通常更近。")

conclusion_text = "\n".join(text)
print(conclusion_text)

with open(os.path.join(RESULT_TABLE_DIR, "g3_conclusion_draft.txt"), "w", encoding="utf-8") as f:
    f.write(conclusion_text)